# Engine Condition - Predictive Maintenance Baseline

Class 508 - SmartOps

Goal: predict `Engine Condition` (0 = Normal, 1 = Needs Maintenance) from six sensor readings, and rank engines by risk so a maintenance team can choose which to inspect first.

Data: `data/engine/engine_data.csv` (19,535 rows, 7 columns). Provenance is unverified (third-party upload); see `reports/collected_data_and_readiness.md`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, RocCurveDisplay)

RANDOM_STATE = 42

import os

LOCAL_PATH = '../data/engine/engine_data.csv'
DATA_URL = 'https://huggingface.co/datasets/jskswamy/predictive-maintenance-data/resolve/main/engine_data.csv'
# Use the local copy when running from the repo; otherwise (e.g. Colab) download from Hugging Face
DATA_PATH = LOCAL_PATH if os.path.exists(LOCAL_PATH) else DATA_URL

## 1. Load and inspect

In [ ]:
df = pd.read_csv(DATA_PATH)
TARGET = 'Engine Condition'
FEATURES = [c for c in df.columns if c != TARGET]
print(df.shape)
display(df.head())
display(df.describe().T)

In [ ]:
print('Missing cells:', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))
print(df[TARGET].value_counts().rename({0: 'Normal', 1: 'Needs Maintenance'}))
print(df[TARGET].value_counts(normalize=True).round(3))

## 2. Data quality flags

Outliers are **flagged, not removed**. Two views: coolant temperature above ~100 C (physically implausible for a liquid-cooled engine) and the 1.5 x IQR rule per sensor. IQR outliers are common in sensor data and are not necessarily errors; section 6 checks whether removing them changes the result.

In [ ]:
SENSORS = FEATURES
q1, q3 = df[SENSORS].quantile(0.25), df[SENSORS].quantile(0.75)
iqr = q3 - q1
iqr_out = (df[SENSORS] < q1 - 1.5 * iqr) | (df[SENSORS] > q3 + 1.5 * iqr)

summary = pd.DataFrame({
    'iqr_outliers': iqr_out.sum(),
    'share': (iqr_out.mean()).round(4),
    'needs_maint_rate': [df.loc[iqr_out[c], TARGET].mean() for c in SENSORS],
})
display(summary)

coolant_hot = df['Coolant Temp'] > 100
any_iqr = iqr_out.any(axis=1)
print('Coolant Temp > 100 C:', int(coolant_hot.sum()), 'rows')
print('Rows with any IQR outlier:', int(any_iqr.sum()), f'({any_iqr.mean():.1%})')
print('Needs-maintenance rate: outlier rows', round(df.loc[any_iqr, TARGET].mean(), 3), '| all rows', round(df[TARGET].mean(), 3))

**Cleaning decision:** keep all 19,535 rows for the baseline (no evidence the outliers are errors). Only the Coolant Temp > 100 C rows are suspicious; the IQR outliers are sensitivity-tested in section 6.

## 3. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), FEATURES):
    for label, name in [(0, 'Normal'), (1, 'Needs Maintenance')]:
        ax.hist(df.loc[df[TARGET] == label, col], bins=40, alpha=0.55, label=name, density=True)
    ax.set_title(col)
axes[0, 0].legend()
plt.tight_layout()
plt.show()

In [ ]:
corr = df.corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
fig.colorbar(im)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

## 4. Train / test split (stratified)

In [ ]:
X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape, y_train.mean().round(3), y_test.mean().round(3))

## 5. Models

In [ ]:
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Gradient Boosting': HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, m in models.items():
    auc = cross_val_score(m, X_train, y_train, cv=cv, scoring='roc_auc')
    f1 = cross_val_score(m, X_train, y_train, cv=cv, scoring='f1')
    test_auc = roc_auc_score(y_test, m.fit(X_train, y_train).predict_proba(X_test)[:, 1])
    rows.append({'model': name, 'cv_auc': auc.mean(), 'cv_f1': f1.mean(), 'test_auc': test_auc})
display(pd.DataFrame(rows).round(4).sort_values('cv_auc', ascending=False))

In [ ]:
best_name = 'Gradient Boosting'  # change after reviewing the CV table
best = models[best_name].fit(X_train, y_train)
proba = best.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
print(best_name)
print('Test ROC AUC:', round(roc_auc_score(y_test, proba), 4))
print(classification_report(y_test, pred, target_names=['Normal', 'Needs Maintenance']))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=['Normal', 'Needs Maint.']).plot(ax=axes[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[1])
plt.tight_layout()
plt.show()

### Feature importance (permutation)

In [ ]:
imp = permutation_importance(best, X_test, y_test, scoring='roc_auc', n_repeats=10, random_state=RANDOM_STATE)
pd.Series(imp.importances_mean, index=FEATURES).sort_values().plot.barh(title='Permutation importance (drop in AUC)')
plt.tight_layout()
plt.show()

## 6. Sensitivity: train without outlier rows

Like-for-like comparison: the IQR bounds come from the **training rows only**, outlier rows are dropped from the training set only, and both models are scored on the **same original test rows**.

In [ ]:
q1_tr, q3_tr = X_train.quantile(0.25), X_train.quantile(0.75)
iqr_tr = q3_tr - q1_tr
lo, hi = q1_tr - 1.5 * iqr_tr, q3_tr + 1.5 * iqr_tr
train_out = ((X_train < lo) | (X_train > hi)).any(axis=1) | (X_train['Coolant Temp'] > 100)

clean_model = clone(models[best_name]).fit(X_train[~train_out], y_train[~train_out])
auc_all = roc_auc_score(y_test, best.predict_proba(X_test)[:, 1])
auc_clean = roc_auc_score(y_test, clean_model.predict_proba(X_test)[:, 1])
print(f'Training rows: {len(X_train)} -> {int((~train_out).sum())} after removing outliers')
print(f'Test ROC AUC ({best_name}), same {len(X_test)} test rows: all training rows {auc_all:.4f} | outliers removed {auc_clean:.4f}')

## 7. Risk ranking for inspection

If the team can inspect only the top *k*% of engines, how many true "Needs Maintenance" cases do they catch?

In [ ]:
rank = pd.DataFrame({'proba': proba, 'y': y_test.values}).sort_values('proba', ascending=False).reset_index(drop=True)
rank['cum_caught'] = rank['y'].cumsum() / rank['y'].sum()
for k in (0.1, 0.2, 0.3, 0.5):
    n = int(len(rank) * k)
    print(f"Inspect top {k:.0%}: catches {rank['cum_caught'].iloc[n-1]:.1%} of cases, precision {rank['y'].iloc[:n].mean():.1%}")
plt.plot(np.arange(1, len(rank) + 1) / len(rank), rank['cum_caught'], label='model')
plt.plot([0, 1], [0, 1], '--', label='random')
plt.xlabel('Share of engines inspected'); plt.ylabel('Share of cases caught'); plt.legend(); plt.title('Cumulative gain')
plt.show()

## 8. Limitations

- Provenance, real vs. synthetic status and label definition are unknown.
- No time or engine ID: no trend or remaining-useful-life modelling.
- Binary label only; no failure cause.
- No cost data; any savings estimate needs stated assumptions.